# USA License-Plate Detection + OCR — Inference

Two-stage pipeline: the finetuned **`best.pt`** detector finds plate boxes, then **easyocr** reads the
characters. Tuned for USA plates (varied per-state formats + vanity plates) with a 5-layer post-processing
strategy — no hard regex, so no valid plate is ever rejected.

**Run order:** cells top-to-bottom. Set `IMAGE_PATH` / `VIDEO_PATH` to your own files.
Outputs (annotated media) are written to `outputs/`. First run downloads easyocr models (~100 MB, one-time).

In [ ]:
import os
from pathlib import Path
from collections import defaultdict, deque, Counter

import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt

from ultralytics import YOLO
import easyocr

In [ ]:
# --- device selection ---
# YOLO runs on Apple's MPS (Metal) GPU when available; easyocr's `gpu` flag only
# understands CUDA, so on a Mac it runs on CPU (plenty fast for small plate crops).
device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print(f"YOLO device: {device}")

# Finetuned single-class license-plate DETECTOR (draws boxes, does not read text)
model = YOLO("best.pt")

# easyocr READER turns each detected plate crop into characters.
# First run downloads ~100 MB of models (one-time, needs internet).
reader = easyocr.Reader(["en"], gpu=torch.cuda.is_available())
print("easyocr ready (CPU)" if not torch.cuda.is_available() else "easyocr ready (CUDA)")

## 1. Configuration

Thresholds and the Layer 3 stopword lists (state names + slogans) that keep OCR focused on the plate number.
Annotated outputs are written to `outputs/`.

In [ ]:
# --- pipeline config ---
ALLOWLIST = "ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789"  # Layer 1: valid plate characters
MIN_LEN, MAX_LEN = 4, 8      # Layer 4: plausible USA plate length
OCR_CONF = 0.30              # Layer 4: minimum mean OCR confidence to keep a read
DET_CONF = 0.25             # YOLO detection confidence threshold
UPSCALE = 2                 # Layer 2: enlarge crops this much before OCR
PAD = 4                     # pixels of context added around each detection box
VOTE_WINDOW = 15            # Layer 5: frames of read history kept per tracked plate

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# Layer 3: distinctive words of the 50 states + DC. On a plate crop, easyocr often
# also reads the state name; we drop any box matching one of these words.
STATE_NAMES = {
    "ALABAMA", "ALASKA", "ARIZONA", "ARKANSAS", "CALIFORNIA", "COLORADO", "CONNECTICUT",
    "DELAWARE", "FLORIDA", "GEORGIA", "HAWAII", "IDAHO", "ILLINOIS", "INDIANA", "IOWA",
    "KANSAS", "KENTUCKY", "LOUISIANA", "MAINE", "MARYLAND", "MASSACHUSETTS", "MICHIGAN",
    "MINNESOTA", "MISSISSIPPI", "MISSOURI", "MONTANA", "NEBRASKA", "NEVADA", "HAMPSHIRE",
    "JERSEY", "MEXICO", "YORK", "CAROLINA", "DAKOTA", "OHIO", "OKLAHOMA", "OREGON",
    "PENNSYLVANIA", "ISLAND", "TENNESSEE", "TEXAS", "UTAH", "VERMONT", "VIRGINIA",
    "WASHINGTON", "WISCONSIN", "WYOMING", "COLUMBIA",
}
# Layer 3: common plate slogans / footer text (compared after spaces are stripped).
SLOGANS = {
    "GOLDENSTATE", "SUNSHINESTATE", "EMPIRESTATE", "GARDENSTATE", "LONESTARSTATE",
    "GRANDCANYONSTATE", "LANDOFLINCOLN", "FIRSTINFLIGHT", "LIVEFREEORDIE",
    "GREATLAKES", "MOUNTRUSHMORE", "AMERICASDAIRYLAND", "DMV", "GOV", "COM", "USA",
}

## 2. OCR pipeline helpers (Layers 1–4)

`read_plate(crop)` is the core: **Layer 1** constrains OCR to plate characters (`allowlist`), **Layer 2**
upscales/enhances the crop, **Layer 3** discards state-name/slogan text boxes, and **Layer 4** normalizes and
gates on length + confidence. No per-state regex, so vanity and out-of-state plates survive.

In [ ]:
def clean_text(s):
    """Uppercase and keep only allowlisted plate characters (drops spaces/punctuation)."""
    return "".join(ch for ch in s.upper() if ch in ALLOWLIST)


def is_stopword(raw):
    """Layer 3: True if an OCR box is a state name or slogan rather than the plate number."""
    up = raw.upper()
    collapsed = "".join(ch for ch in up if ch in ALLOWLIST)
    if collapsed in SLOGANS:
        return True
    for word in up.replace("-", " ").split():
        wc = "".join(ch for ch in word if ch in ALLOWLIST)
        if wc in STATE_NAMES:
            return True
    return False


def preprocess_crop(crop):
    """Layer 2: upscale + grayscale + CLAHE contrast to make small plates readable."""
    if crop.size == 0:
        return crop
    g = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    if UPSCALE != 1:
        g = cv2.resize(g, None, fx=UPSCALE, fy=UPSCALE, interpolation=cv2.INTER_CUBIC)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    return clahe.apply(g)


def read_plate(crop):
    """Layers 1-4: OCR one plate crop. Returns (plate_str, confidence) or None."""
    proc = preprocess_crop(crop)
    if proc.size == 0:
        return None

    # Layer 1: allowlist constrains the recognizer to plate characters only.
    results = reader.readtext(proc, allowlist=ALLOWLIST, detail=1)

    boxes = []
    for bbox, text, conf in results:
        if is_stopword(text):            # Layer 3
            continue
        cleaned = clean_text(text)
        if not cleaned:
            continue
        ys = [p[1] for p in bbox]
        xs = [p[0] for p in bbox]
        boxes.append({
            "text": cleaned, "conf": float(conf),
            "h": max(ys) - min(ys), "yc": (max(ys) + min(ys)) / 2, "x": min(xs),
        })
    if not boxes:
        return None

    # The plate number is the tallest text line; keep boxes on that same line and
    # read them left-to-right (handles reads split like "7ABC" + "123").
    ref = max(boxes, key=lambda b: b["h"])
    line = [b for b in boxes if abs(b["yc"] - ref["yc"]) <= 0.5 * ref["h"]]
    line.sort(key=lambda b: b["x"])
    plate = "".join(b["text"] for b in line)
    conf = float(np.mean([b["conf"] for b in line]))

    # Layer 4: length + confidence gate (no per-state regex — keeps vanity plates).
    if not (MIN_LEN <= len(plate) <= MAX_LEN) or conf < OCR_CONF:
        return None
    return plate, conf


def annotate(frame, xyxy, label):
    """Draw a detection box and its plate label on the frame (in place)."""
    x1, y1, x2, y2 = map(int, xyxy)
    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 200, 0), 2)
    if label:
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.8, 2)
        cv2.rectangle(frame, (x1, y1 - th - 8), (x1 + tw + 6, y1), (0, 200, 0), -1)
        cv2.putText(frame, label, (x1 + 3, y1 - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 2)
    return frame

## 3. Single-image inference

Detect plates with `best.pt`, read each with the Layer 1–4 pipeline, draw the result, save an annotated PNG
to `outputs/`, and show it inline. Set `IMAGE_PATH` to your own photo.

In [ ]:
def run_image(path, save=True, show=True):
    """Detect + OCR every plate in one image. Returns a list of (plate, conf)."""
    img = cv2.imread(str(path))
    if img is None:
        raise FileNotFoundError(f"Could not read image: {path}")

    result = model(img, conf=DET_CONF, device=device, verbose=False)[0]
    plates = []
    for box in result.boxes:
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        x1p, y1p = max(0, int(x1) - PAD), max(0, int(y1) - PAD)
        x2p = min(img.shape[1], int(x2) + PAD)
        y2p = min(img.shape[0], int(y2) + PAD)
        read = read_plate(img[y1p:y2p, x1p:x2p])
        label = ""
        if read:
            plate, conf = read
            plates.append((plate, conf))
            label = f"{plate} {conf:.2f}"
        annotate(img, (x1, y1, x2, y2), label)

    if save:
        out = OUTPUT_DIR / f"annotated_{Path(path).stem}.png"
        cv2.imwrite(str(out), img)
        print(f"saved -> {out}")
    if show:
        plt.figure(figsize=(12, 8))
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        plt.axis("off")
        plt.show()
    print("Plates:", plates if plates else "none detected")
    return plates


# --- run on a single image: point IMAGE_PATH at your own USA-car photo ---
IMAGE_PATH = "test.jpg"
if Path(IMAGE_PATH).exists():
    run_image(IMAGE_PATH)
else:
    print(f"Set IMAGE_PATH to a real image. '{IMAGE_PATH}' not found.")

## 4. Video inference — tracking + temporal voting (Layer 5)

`model.track()` gives every plate a persistent id across frames. We accumulate each id's OCR reads in a
`deque` and output the **majority-vote** string, so a plate misread in a few blurry frames is corrected by
the frames that read it right. Writes an annotated `.mp4` to `outputs/` and shows sample frames inline.

In [ ]:
def vote(reads):
    """Layer 5: majority-vote a track's plate reads, tie-broken by summed confidence."""
    if not reads:
        return None
    counts, scores = Counter(), defaultdict(float)
    for plate, conf in reads:
        counts[plate] += 1
        scores[plate] += conf
    return max(counts, key=lambda p: (counts[p], scores[p]))


def run_video(path, save=True, max_frames=None, show_samples=6):
    """Detect + track + OCR plates across a video. Each tracked plate's reads are
    accumulated and majority-voted so the final label is stable across frames.
    Returns {track_id: voted_plate}."""
    path = str(path)
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        raise FileNotFoundError(f"Could not open video: {path}")
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()

    writer = None
    out_path = OUTPUT_DIR / f"annotated_{Path(path).stem}.mp4"
    if save:
        writer = cv2.VideoWriter(str(out_path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

    track_reads = defaultdict(lambda: deque(maxlen=VOTE_WINDOW))  # id -> recent (plate, conf)
    samples = []

    stream = model.track(source=path, stream=True, persist=True, conf=DET_CONF,
                         device=device, tracker="bytetrack.yaml", verbose=False)
    for fi, result in enumerate(stream):
        if max_frames and fi >= max_frames:
            break
        frame = result.orig_img.copy()
        boxes = result.boxes
        if boxes is not None and boxes.id is not None:
            for box, tid in zip(boxes, boxes.id.int().tolist()):
                x1, y1, x2, y2 = box.xyxy[0].tolist()
                x1p, y1p = max(0, int(x1) - PAD), max(0, int(y1) - PAD)
                x2p = min(frame.shape[1], int(x2) + PAD)
                y2p = min(frame.shape[0], int(y2) + PAD)
                read = read_plate(frame[y1p:y2p, x1p:x2p])
                if read:
                    track_reads[tid].append(read)
                voted = vote(track_reads[tid])
                annotate(frame, (x1, y1, x2, y2), f"#{tid} {voted}" if voted else f"#{tid}")
        if writer is not None:
            writer.write(frame)
        if len(samples) < show_samples and fi % 15 == 0:
            samples.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

    if writer is not None:
        writer.release()
        print(f"saved -> {out_path}")

    final = {tid: vote(reads) for tid, reads in track_reads.items()}
    final = {tid: p for tid, p in final.items() if p}
    print("\nFinal plate per tracked car:")
    for tid, plate in final.items():
        print(f"  car #{tid}: {plate}")

    if samples:
        cols = 3
        rows = (len(samples) + cols - 1) // cols
        plt.figure(figsize=(15, 4 * rows))
        for i, s in enumerate(samples):
            plt.subplot(rows, cols, i + 1)
            plt.imshow(s)
            plt.axis("off")
        plt.tight_layout()
        plt.show()
    return final


# --- run on a video: point VIDEO_PATH at your own clip ---
VIDEO_PATH = "test.mp4"
if Path(VIDEO_PATH).exists():
    run_video(VIDEO_PATH)
else:
    print(f"Set VIDEO_PATH to a real video. '{VIDEO_PATH}' not found.")

## 5. Results summary

`summarize(...)` prints collected plates as a table. Run the image/video cells above first, then aggregate their return values.

In [ ]:
def summarize(results):
    """Pretty-print a {source_or_track: (plate, conf) | plate} mapping as a table."""
    print(f"{'source':<16}{'plate':<12}{'conf':>6}")
    print("-" * 34)
    for key, val in results.items():
        if isinstance(val, tuple):
            plate, conf = val
            print(f"{str(key):<16}{plate:<12}{conf:>6.2f}")
        else:
            print(f"{str(key):<16}{str(val):<12}")


# Example workflow — run the inference cells above first, then aggregate:
#   img_plates = run_image(IMAGE_PATH, show=False)          # list of (plate, conf)
#   vid_plates = run_video(VIDEO_PATH)                       # {track_id: plate}
#   summarize({f"img_{i}": p for i, p in enumerate(img_plates)})
#   summarize({f"car#{tid}": plate for tid, plate in vid_plates.items()})
print("Pipeline ready. Use run_image(path) / run_video(path), then summarize(...).")